# XRD analysis

Moved out of `Data_processing.ipynb`. The cells below still read structures from the siman database.


In [ ]:
import sys
from pathlib import Path

def _repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "plot_colors.py").is_file():
            return cand
    raise FileNotFoundError("plot_colors.py not found from " + str(here))

sys.path.insert(0, str(_repo_root()))
from plot_colors import series_colors


In [2]:
import os, re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
# from pymatgen.ext.matproj import MPRester
from mp_api.client import MPRester
from pymatgen.core import Composition, Element
from pymatgen.analysis.phase_diagram import GrandPotentialPhaseDiagram, PhaseDiagram
from pymatgen.analysis.interface_reactions import InterfacialReactivity, GrandPotentialInterfacialReactivity
from emmet.core.thermo import ThermoType
from pymatgen.entries.computed_entries import ComputedEntry
from matplotlib.patches import Patch
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pathlib import Path



In [3]:
%%capture
import siman #program package to manage DFT calculations https://github.com/dimonaks/siman
from siman.calc_manage import smart_structure_read, get_structure_from_matproj, get_structure_from_matproj_new
from siman.calc_manage import add, res
# Update configurations
from siman import header
from siman.database import write_database, read_database
from siman.set_functions import read_vasp_sets
from siman.header import db
from siman.header import _update_configuration
_update_configuration('project_conf.py')
read_database() # read saved database if available
from pydoc import importfile
project_sets = importfile('project_sets.py')
varset = read_vasp_sets(project_sets.user_vasp_sets, override_global = 0) #read user sets

from siman import thermo

# header.PATH2PROJECT = '../dft_calculations/'
header.PATH2EDITOR = 'notepad.exe'
header.PATH2POTENTIALS = "/home/a.burov/soft/vasp_potentials/potpaw_PBE_MPIE/"

from matplotlib import rc



In [4]:
from ase.optimize import FIRE  
from ase.io import read, write
from ase.visualize import view


## XRD profiles

In [ ]:
from pymatgen.core import Structure


In [ ]:
data_xrd_path = "/home/a.burov/icys_2025/niohf/data/NiOHF_clean_#01.gdf"

In [ ]:
with open(data_xrd_path, 'r', encoding='latin-1') as f:
    lines = f.readlines() 


In [ ]:
for idx, line in enumerate(lines):
    if '<beginc><endc>' in line:
        print(f"Found at line {idx}: {line.strip()}")
        break


In [ ]:
idx

In [ ]:
angle_min = float(lines[idx+1])
angle_max = float(lines[idx+2])
step = float(lines[idx+3])


In [ ]:
int_xrd = [int(inten) for inten in lines[idx+7:]]

In [ ]:
angles_xrd = [angle_min + (entry_num-idx-7)*step for entry_num in range(idx+7, len(lines), 1) ]


In [ ]:
# calculator = XRDCalculator(wavelength="CuKa", symprec=1e-10, 
#                debye_waller_factors={"Ni": 0.3, "O": 0.5, "H": 1.5, "F": 0.4})  # Cu Kα radiation (1.5406 Å)

calculator = XRDCalculator(wavelength="CoKa1", symprec=1e-10, 
               debye_waller_factors={"Ni": 0.3, "O": 0.5, "H": 1.5, "F": 0.4})  # Cu Kα radiation (1.5406 Å)


# Ni	0.3 - 0.5	Heavy transition metal, low thermal motion
# O	0.5 - 0.8	Moderate thermal motion in oxide framework
# H	1.5 - 2.5	Lightest atom, highest thermal displacement
# F	0.4 - 0.7  Similar to O, but slightly lower in fluorides


In [ ]:
# latt_scale_factor = 0.87
latt_scale_factor = 1.0


In [ ]:
structure_diaspore = db['delta.sc', '9bulk_eos', 100].copy().end
latt = np.array(structure_diaspore.rprimd)
structure_diaspore.rprimd = latt * latt_scale_factor
structure_diaspore.update_xcart()
structure_diaspore.update_xred()


In [ ]:
structure_diaspore = structure_diaspore.convert2pymatgen()
pattern_diaspore = calculator.get_pattern(structure_diaspore, two_theta_range=(0, 100))


In [ ]:
structure_layered = db["layered", '9bulk', 1].copy().end
structure_layered = structure_layered.get_conventional_cell()
latt = np.array(structure_layered.rprimd)
structure_layered.rprimd = latt * latt_scale_factor
structure_layered.update_xcart()
structure_layered.update_xred()



In [ ]:
structure_layered = structure_layered.convert2pymatgen()

pattern_layered = calculator.get_pattern(structure_layered, two_theta_range=(0, 100))


In [ ]:
st_layered = Structure.from_file("/home/a.burov/icys_2025/niohf/optimized/sevenn/layered.cif")

In [ ]:
# structure_layered = db["layered", '9bulk', 1].copy().end
# structure_layered.write_cif("/home/a.burov/icys_2025/niohf/data/layered_dft.cif")

In [ ]:
int_list = pattern_layered.as_dict()["y"]
theta_list = pattern_layered.as_dict()["x"]

hkl_list = []
for mill_idx in pattern_layered.as_dict()["hkls"]: 
    hkl_cur = mill_idx[0]['hkl'] 
    hkl_cur = [str(hkl) for hkl in hkl_cur ]
    hkl_cur = "".join(hkl_cur)
    hkl_list.append(hkl_cur)


In [ ]:
fontsize = 14
lw = 2.0
width = 0.3

fig, ax = plt.subplots(1, 1, figsize=(7, 4), dpi=600)
plt.tight_layout()


# Add titles and labels
ax.set_ylabel(r'Intensity', fontsize=fontsize)
ax.set_xlabel(r'$2\Theta$, degrees', fontsize=fontsize)

ax.tick_params(axis='both', which='major', labelsize=fontsize)
ax.tick_params(axis='both', which='minor', labelsize=fontsize-2)
ax.yaxis.get_offset_text().set_fontsize(10)
ax.xaxis.set_tick_params(width=2, length=7)
ax.yaxis.set_tick_params(width=2, length=7)

shift_angle = 0
c_exp, c_lay, c_dia = series_colors(3)
ax.plot(angles_xrd, np.array(int_xrd)/max(int_xrd), color=c_exp, label='experiment')
plt.bar(pattern_layered.x + shift_angle, pattern_layered.y/max(pattern_layered.y), width=0.3, color=c_lay, label='layered')
plt.bar(pattern_diaspore.x + shift_angle, pattern_diaspore.y/max(pattern_diaspore.y), width=0.3, color=c_dia, label='diaspore')

plt.legend(fontsize=fontsize-4, loc=1)


fig.tight_layout()
fig.savefig("/home/a.burov/icys_2025/niohf/data/figures/xrd_both.png", dpi=450)



## Single-phase sample

In [ ]:
exp_data = np.loadtxt("/home/a.burov/icys_2025/niohf/data/Ni(OH)F_8h_50ml.xy")
# Assign to arrays
angles_list = exp_data[:, 0]  # First column
int_list = exp_data[:, 1]     # Second column

In [ ]:
calculator = XRDCalculator(wavelength="CuKa1", symprec=1e-10, 
               debye_waller_factors={"Ni": 0.3, "O": 0.5, "H": 1.5, "F": 0.4})  # Cu Kα radiation (1.5406 Å)


In [ ]:
structure_layered = db["layered", '9bulk', 1].copy().end
# structure_layered = structure_layered.get_conventional_cell()
structure_layered = structure_layered.get_primitive_cell()


In [ ]:
structure_layered = structure_layered.convert2pymatgen()


In [ ]:
pattern_layered = calculator.get_pattern(structure_layered, two_theta_range=(0, 100))


In [ ]:
hkl_list = []
for mill_idx in pattern_layered.as_dict()["hkls"]: 
    hkl_cur = mill_idx[0]['hkl'] 
    hkl_cur = [str(hkl) for hkl in hkl_cur ]
    hkl_cur = "".join(hkl_cur)
    hkl_list.append(hkl_cur)


In [ ]:
hkl_list

In [ ]:
fontsize = 14
lw = 2.0
width = 0.3

fig, ax = plt.subplots(1, 1, figsize=(7, 4), dpi=600)
plt.tight_layout()


# Add titles and labels
ax.set_ylabel(r'Intensity', fontsize=fontsize)
ax.set_xlabel(r'$2\Theta$, degrees', fontsize=fontsize)

ax.tick_params(axis='both', which='major', labelsize=fontsize)
ax.tick_params(axis='both', which='minor', labelsize=fontsize-2)
ax.yaxis.get_offset_text().set_fontsize(10)
ax.xaxis.set_tick_params(width=2, length=7)
ax.yaxis.set_tick_params(width=2, length=7)

shift_angle = 0
c_exp, c_lay = series_colors(2)
ax.plot(angles_list, np.array(int_list)/max(int_list), color=c_exp, label='experiment', zorder=1)
plt.bar(pattern_layered.x + shift_angle, pattern_layered.y/max(pattern_layered.y), width=0.2, color=c_lay, label='layered', zorder=2)

plt.legend(fontsize=10, loc=1)


fig.tight_layout()
fig.savefig("/home/a.burov/icys_2025/niohf/data/figures/xrd_layered.png", dpi=450)


### LNO test with Co source

In [ ]:
import pymatgen as pm
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.io.ase import AseAtomsAdaptor


In [ ]:
exp_data = np.loadtxt("/home/a.burov/icys_2025/niohf/data/LNO1_A.xyd")
# Assign to arrays
angles_list = exp_data[:, 0]  # First column
int_list = exp_data[:, 1]     # Second column

In [ ]:
st_ase = read("/home/a.burov/icys_2025/niohf/data/LNO.cif")


In [ ]:
structure_lno = AseAtomsAdaptor.get_structure(st_ase)


In [ ]:
structure_lno.get_space_group_info()

In [ ]:
xrd = XRDCalculator(wavelength="CoKa") #initiate XRD calculator (can specify various options here)
structure_lno_pattern = xrd.get_pattern(structure_lno, scaled=True, two_theta_range=(0, 100))


In [ ]:
structure_lno_pattern

In [ ]:
hkl_list = []
for mill_idx in structure_lno_pattern.as_dict()["hkls"]: 
    hkl_cur = mill_idx[0]['hkl'] 
    hkl_cur = [str(hkl) for hkl in hkl_cur ]
    hkl_cur = "".join(hkl_cur)
    hkl_list.append(hkl_cur)

print(hkl_list)




In [ ]:
fontsize = 14
lw = 2.0
width = 0.3

fig, ax = plt.subplots(1, 1, figsize=(7, 5), dpi=600)
plt.tight_layout()


# Add titles and labels
ax.set_ylabel(r'Intensity', fontsize=fontsize)
ax.set_xlabel(r'$2\Theta$, degrees', fontsize=fontsize)

ax.tick_params(axis='both', which='major', labelsize=fontsize)
ax.tick_params(axis='both', which='minor', labelsize=fontsize-2)
ax.yaxis.get_offset_text().set_fontsize(10)
ax.xaxis.set_tick_params(width=2, length=7)
ax.yaxis.set_tick_params(width=2, length=7)

shift_angle = 0
c_exp, c_lno = series_colors(2)
ax.plot(angles_list, np.array(int_list)/max(int_list), color=c_exp, label='experiment', zorder=1)
plt.bar(structure_lno_pattern.x + shift_angle, structure_lno_pattern.y/max(structure_lno_pattern.y), width=0.2, color=c_lno, label='LNO', zorder=2)

# plt.plot(tt, I, width=0.2, color='red', label='layered', zorder=2)

plt.legend(fontsize=10, loc=1)


fig.tight_layout()
fig.savefig("/home/a.burov/icys_2025/niohf/data/figures/xrd_lno.png", dpi=450)


